In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('/content/qoute_dataset.csv')

In [5]:
quotes = df['quote']
quotes = quotes.str.lower()


In [6]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [9]:
word_index = tokenizer.word_index
print(f"Found {len(word_index)} unique tokens.")
list(word_index.items())[:10]

Found 8978 unique tokens.


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [10]:
sequences = tokenizer.texts_to_sequences(quotes)


In [11]:
x=[]
y=[]
for sequence in sequences:
    for i in range(1, len(sequence)):
        x.append(sequence[:i])
        y.append(sequence[i])

In [12]:
max_len = max([len(seq) for seq in x])

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(x, maxlen=max_len, padding='pre')
Y_padded = np.array(y, dtype=np.int32)

In [14]:
from tensorflow.keras.utils import to_categorical
y_one_hot_encoded = to_categorical(Y_padded, num_classes=vocab_size)

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [16]:
embedding_dim = 100
rnn_units = 128

In [17]:

rnn_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    SimpleRNN(units=rnn_units),
    Dense(units=vocab_size, activation="softmax")
])

rnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    LSTM(units=rnn_units),
    Dense(units=vocab_size, activation="softmax")
])
lstm_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
lstm_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
epochs = 10
batch_size = 128

history_rnn = rnn_model.fit(
    X_padded,
    y_one_hot_encoded,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)


Epoch 1/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 48s 81ms/step - accuracy: 0.0441 - loss: 6.7255 - val_accuracy: 0.0595 - val_loss: 6.6067
Epoch 2/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.0805 - loss: 6.0831 - val_accuracy: 0.0916 - val_loss: 6.4160
Epoch 3/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.1074 - loss: 5.6890 - val_accuracy: 0.1028 - val_loss: 6.3335
Epoch 4/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.1266 - loss: 5.3663 - val_accuracy: 0.1093 - val_loss: 6.3628
Epoch 5/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 72ms/step - accuracy: 0.1441 - loss: 5.0798 - val_accuracy: 0.1146 - val_loss: 6.4260
Epoch 6/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.1604 - loss: 4.8200 - val_accuracy: 0.1123 - val_loss: 6.4902
Epoch 7/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.1779 - loss: 4.5773 - val_accuracy: 0.1121 - val_loss: 6.5654
Epoch 8/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.1994 - loss: 4.3478 - 

In [22]:
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot_encoded,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

Epoch 1/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 40s 65ms/step - accuracy: 0.0417 - loss: 6.7576 - val_accuracy: 0.0423 - val_loss: 6.7350
Epoch 2/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 63ms/step - accuracy: 0.0587 - loss: 6.2918 - val_accuracy: 0.0680 - val_loss: 6.5967
Epoch 3/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 62ms/step - accuracy: 0.0856 - loss: 6.0202 - val_accuracy: 0.0875 - val_loss: 6.5095
Epoch 4/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 63ms/step - accuracy: 0.1045 - loss: 5.7861 - val_accuracy: 0.0957 - val_loss: 6.4470
Epoch 5/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 62ms/step - accuracy: 0.1174 - loss: 5.5860 - val_accuracy: 0.1020 - val_loss: 6.4664
Epoch 6/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 62ms/step - accuracy: 0.1287 - loss: 5.4100 - val_accuracy: 0.1065 - val_loss: 6.4793
Epoch 7/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 62ms/step - accuracy: 0.1371 - loss: 5.2468 - val_accuracy: 0.1082 - val_loss: 6.5077
Epoch 8/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 62ms/step - accuracy: 0.1463 - loss: 5.0940 - 

In [25]:
lstm_model.save("lstm_model.h5")
rnn_model.save("rnn_model.h5")

In [26]:
index_to_word ={}
for word , index in word_index.items():
  index_to_word[index] = word

In [29]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen=max_len,padding='pre')
  prediction = model.predict(seq,verbose=0)
  predicted_index = np.argmax(prediction)
  output_word = index_to_word[predicted_index]
  return output_word

In [36]:
seed_text = "what is"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)


not


In [53]:
def generate_text(model,tokenizer,seed_text,n_words,max_len):
  generated_sequence = seed_text
  for _ in range(n_words):
    # Prepare the input sequence for the predictor
    # Take the last max_len-1 words from the current generated_sequence
    # to ensure the input length matches the model's expected input_length
    current_seq_tokens = generated_sequence.lower().split()
    input_for_predictor = " ".join(current_seq_tokens[-(max_len-1):])

    next_word = predictor(model,tokenizer,input_for_predictor,max_len)
    if next_word == "":
        break
    generated_sequence += " " + next_word
  return generated_sequence

In [54]:
seed_phrase = "The quick brown fox"
generated_sequence = generate_text(lstm_model, tokenizer, seed_phrase, 20, max_len)
print(generated_sequence)

The quick brown fox with the world is a man who was a woman i have to be a woman you can be a


In [55]:
import pickle
with open("tokenizer.pkl","wb") as f:
  pickle.dump(tokenizer,f)

In [56]:
with open("max_len.pkl","wb") as f:
  pickle.dump(max_len,f)